# SWAN-SF: Prepare Clean Train/Test Splits (Tabular Baselines)

This notebook verifies that the cleaned SWAN-SF data is available, performs additional cleanliness checks,
defines features/target for tabular models, and materializes explicit train/test (and optional validation)
splits for use in downstream modeling notebooks (e.g., Logistic Regression and XGBoost baselines).

## 1. Configure Paths and Parameters

We point to the cleaned SWAN-SF dataset directory already used by the project (`data/Cleaned SWANSF Dataset/`).
For the tabular baselines, we will:
- Load the partitioned train/test arrays using the existing loader utilities.
- Optionally persist flattened/tabular versions of these splits for use in other notebooks.

Key parameters include:
- `BASE_DIR`: root directory of the cleaned dataset.
- `random_state`: for reproducible downstream splits (if we create additional validation splits).
- `use_top_features_only`: whether to restrict to the top-5 features from EDA.

In [5]:
# Configure paths and core parameters
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Walk up from the current working directory until src/sdp is found."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "sdp").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root containing 'src/sdp' starting from {start}"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "Cleaned SWANSF Dataset"
OUTPUT_DIR = PROJECT_ROOT / "data" / "prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random_state = 42
use_top_features_only = True  # restrict to strongest univariate features for initial baselines

DATA_DIR, OUTPUT_DIR

(PosixPath('/Users/chintan/Drive/Projects/sdp-solar-flares-classifier/data/Cleaned SWANSF Dataset'),
 PosixPath('/Users/chintan/Drive/Projects/sdp-solar-flares-classifier/data/prepared'))

## 2. Load and Inspect Raw Dataset (Cleaned Partitions)

We re-use the `sdp.data_loader.load_swansf_partitions` helper to load the cleaned train and test
partitions. These are already preprocessed (imputation, normalization, resampling) as per the
dataset documentation.

We will:
- Load all 5 train and 5 test partitions.
- Print basic shape and class balance summaries.

In [6]:
# Ensure src/ is on the Python path when running from the notebook
import sys
import os

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sdp.data_loader import (
    load_swansf_partitions,
    concat_partitions,
    summarize_partitions,
    ATTRIBUTES,
)

print("Data directory exists:", DATA_DIR.exists())
print("Train dir:", (DATA_DIR / "train").exists(), "Test dir:", (DATA_DIR / "test").exists())

Data directory exists: True
Train dir: True Test dir: True


In [7]:
# Load cleaned train and test partitions
X_train_parts, y_train_parts = load_swansf_partitions(str(DATA_DIR), split="train", num_partitions=5)
X_test_parts, y_test_parts = load_swansf_partitions(str(DATA_DIR), split="test", num_partitions=5)

print("Train summary:")
print(summarize_partitions(X_train_parts, y_train_parts))
print("\nTest summary:")
print(summarize_partitions(X_test_parts, y_test_parts))

Train summary:
Partition 1: X=(18773, 60, 24), y=(18773,), pos=8778, neg=9995
Partition 2: X=(19807, 60, 24), y=(19807,), pos=9807, neg=10000
Partition 3: X=(19965, 60, 24), y=(19965,), pos=9968, neg=9997
Partition 4: X=(19320, 60, 24), y=(19320,), pos=9320, neg=10000
Partition 5: X=(19899, 60, 24), y=(19899,), pos=9900, neg=9999
Total samples across partitions: 97764

Test summary:
Partition 1: X=(73492, 60, 24), y=(73492,), pos=1254, neg=72238
Partition 2: X=(88557, 60, 24), y=(88557,), pos=1401, neg=87156
Partition 3: X=(42510, 60, 24), y=(42510,), pos=1424, neg=41086
Partition 4: X=(51261, 60, 24), y=(51261,), pos=1165, neg=50096
Partition 5: X=(75365, 60, 24), y=(75365,), pos=990, neg=74375
Total samples across partitions: 331185


## 3. Validate Data Cleanliness

The partition files are numpy arrays, already cleaned and imputed. We nonetheless perform basic
sanity checks:
- Finite values (no NaN/Inf).
- Expected dimensionality: `(n_samples, n_timesteps, n_attributes)`.
- Matching attribute count with `ATTRIBUTES`.

In [8]:
import numpy as np

def check_cleanliness(X_parts, y_parts, name: str):
    print(f"=== {name} ===")
    for i, (X, y) in enumerate(zip(X_parts, y_parts), start=1):
        n, t, a = X.shape
        print(f"Partition {i}: X shape={X.shape}, y shape={y.shape}")
        assert a == len(ATTRIBUTES), f"Unexpected attribute count in {name} partition {i}"
        assert y.ndim == 1 and y.shape[0] == n, f"Label shape mismatch in {name} partition {i}"
        # Basic NaN/Inf checks
        if not np.isfinite(X).all():
            raise ValueError(f"Non-finite values detected in {name} partition {i} features")
        if not np.isfinite(y).all():
            raise ValueError(f"Non-finite values detected in {name} partition {i} labels")
    print("All basic cleanliness checks passed.\n")

check_cleanliness(X_train_parts, y_train_parts, "Train")
check_cleanliness(X_test_parts, y_test_parts, "Test")

=== Train ===
Partition 1: X shape=(18773, 60, 24), y shape=(18773,)
Partition 2: X shape=(19807, 60, 24), y shape=(19807,)
Partition 3: X shape=(19965, 60, 24), y shape=(19965,)
Partition 4: X shape=(19320, 60, 24), y shape=(19320,)
Partition 5: X shape=(19899, 60, 24), y shape=(19899,)
All basic cleanliness checks passed.

=== Test ===
Partition 1: X shape=(73492, 60, 24), y shape=(73492,)
Partition 2: X shape=(88557, 60, 24), y shape=(88557,)
Partition 3: X shape=(42510, 60, 24), y shape=(42510,)
Partition 4: X shape=(51261, 60, 24), y shape=(51261,)
Partition 5: X shape=(75365, 60, 24), y shape=(75365,)
All basic cleanliness checks passed.



## 4. Apply Final Cleaning Steps (If Needed)

Because SWAN-SF cleaned partitions are already preprocessed, we don't expect to apply further
cleaning here. This section is left as a hook in case additional filters are needed later (e.g.,
dropping extremely short sequences or rare label values).

For now, no extra cleaning is applied; we proceed with the loaded arrays.

## 5. Define Features and Target for Tabular Models

For Logistic Regression and XGBoost baselines, we will transform each time series into a fixed-length
tabular feature vector using simple summary statistics over time.

We start from the full set of attributes defined in `ATTRIBUTES`, but can optionally restrict to the
top-5 features identified in the EDA notebook:
- `SAVNCPP`, `ABSNJZH`, `R_VALUE`, `EPSZ`, `TOTFZ`.

Each timeseries will be summarized by, for example, its mean, standard deviation, min, max, and last
value over the time dimension. This produces a 2D feature matrix suitable for tabular models.

In [9]:
# Define which attributes to use for tabular baselines
TOP5_FEATURES = ["SAVNCPP", "ABSNJZH", "R_VALUE", "EPSZ", "TOTFZ"]

if use_top_features_only:
    feature_names_time = TOP5_FEATURES
else:
    feature_names_time = list(ATTRIBUTES)

feature_indices = [ATTRIBUTES.index(name) for name in feature_names_time]
feature_names_time

['SAVNCPP', 'ABSNJZH', 'R_VALUE', 'EPSZ', 'TOTFZ']

In [10]:
def timeseries_to_tabular(X_parts, y_parts, feature_indices):
    """Flatten list of 3D partitions into a 2D tabular matrix using simple stats.

    For each sample and selected attribute, we compute:
    - mean over time
    - standard deviation over time
    - min over time
    - max over time
    - last observed value
    """
    X_all_list = []
    y_all_list = []
    for X, y in zip(X_parts, y_parts):
        # Select attributes
        X_sel = X[:, :, feature_indices]  # (n, t, k)
        mean_ = X_sel.mean(axis=1)
        std_ = X_sel.std(axis=1)
        min_ = X_sel.min(axis=1)
        max_ = X_sel.max(axis=1)
        last_ = X_sel[:, -1, :]
        # Concatenate along feature axis
        X_tab = np.concatenate([mean_, std_, min_, max_, last_], axis=1)
        X_all_list.append(X_tab)
        y_all_list.append(y)
    X_all = np.concatenate(X_all_list, axis=0)
    y_all = np.concatenate(y_all_list, axis=0)
    return X_all, y_all

X_train_tab, y_train_tab = timeseries_to_tabular(X_train_parts, y_train_parts, feature_indices)
X_test_tab, y_test_tab = timeseries_to_tabular(X_test_parts, y_test_parts, feature_indices)

X_train_tab.shape, X_test_tab.shape

((97764, 25), (331185, 25))

## 6. Perform Train/Test (and Optional Validation) Split

The SWAN-SF dataset already provides a conceptual train/test split via separate partition files.
For the tabular baselines, we keep this high-level split as-is. Optionally, we can carve out a
small validation set from the training portion for hyperparameter tuning.

In [11]:
from sklearn.model_selection import train_test_split

# Optional: create a held-out validation split from the training tabular data
val_size = 0.2  # fraction of training data to use for validation

X_train_tab_sub, X_val_tab, y_train_tab_sub, y_val_tab = train_test_split(
    X_train_tab,
    y_train_tab,
    test_size=val_size,
    random_state=random_state,
    stratify=y_train_tab,
)

X_train_tab_sub.shape, X_val_tab.shape

((78211, 25), (19553, 25))

## 7. Persist Prepared Datasets for Next Notebook

We save the tabular train/validation/test splits to disk so that a separate modeling notebook
can load them directly (without re-running the preparation steps). CSV is sufficient here, but
Parquet or NumPy binary files could also be used.

In [12]:
import pandas as pd

def build_feature_column_names(base_names):
    stats = ["mean", "std", "min", "max", "last"]
    cols = []
    for stat in stats:
        for name in base_names:
            cols.append(f"{name}_{stat}")
    return cols

base_feature_names = [ATTRIBUTES[idx] for idx in feature_indices]
tabular_feature_cols = build_feature_column_names(base_feature_names)

train_df = pd.DataFrame(X_train_tab_sub, columns=tabular_feature_cols)
train_df["label"] = y_train_tab_sub

val_df = pd.DataFrame(X_val_tab, columns=tabular_feature_cols)
val_df["label"] = y_val_tab

test_df = pd.DataFrame(X_test_tab, columns=tabular_feature_cols)
test_df["label"] = y_test_tab

train_path = OUTPUT_DIR / "swansf_tabular_train.csv"
val_path = OUTPUT_DIR / "swansf_tabular_val.csv"
test_path = OUTPUT_DIR / "swansf_tabular_test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, val_path, test_path

(PosixPath('/Users/chintan/Drive/Projects/sdp-solar-flares-classifier/data/prepared/swansf_tabular_train.csv'),
 PosixPath('/Users/chintan/Drive/Projects/sdp-solar-flares-classifier/data/prepared/swansf_tabular_val.csv'),
 PosixPath('/Users/chintan/Drive/Projects/sdp-solar-flares-classifier/data/prepared/swansf_tabular_test.csv'))